# 🚀 Face Recognition Dataset - VGGFace2 Ready for Training

This notebook handles:
1. **Dataset Download** - Download pre-processed VGGFace2-112x112 dataset
2. **Data Verification** - Verify dataset structure and quality
3. **Dataset Analysis** - Analyze identities, splits, and statistics
4. **Usage Preparation** - Prepare for training notebook integration

## 📊 Why VGGFace2-112x112?

| Feature | Original VGGFace2 | This Dataset (112x112) |
|---------|------------------|------------------------|
| Preprocessing | Required | ✅ Already done |
| Face Detection | Manual | ✅ Already aligned |
| Image Size | Variable | ✅ Fixed 112x112 |
| Alignment | Required | ✅ Already aligned |
| Ready to Train | No | ✅ Yes |

## 📦 Dataset: VGGFace2-112x112

| Property | Value |
|----------|-------|
| Images | ~3.3M+ |
| Identities | 9,131 |
| Format | 112x112 RGB JPG |
| Alignment | Pre-aligned faces |
| Split | Train/Test included |
| Source | yakhyokhuja/vggface2-112x112 |

## ⏱️ Time Estimates

| Step | Time |
|------|------|
| Dataset Download | 10-30 min (first time) |
| Verification | 2-5 min |
| Analysis | 1-3 min |
| **Total** | **~15-40 minutes (one-time)** |
| Subsequent use | Seconds (cached) |

## ✨ Key Benefits

✅ **Pre-processed**: Images already aligned and resized to 112x112
✅ **Training-ready**: No additional preprocessing needed
✅ **Large-scale**: 9,131 identities for robust training
✅ **Standard format**: Compatible with MobileFaceNet and similar architectures
✅ **Kaggle-hosted**: Fast downloads with automatic caching

## 1. 📦 Environment Setup

In [19]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✅ Running on Google Colab")
    # Mount Google Drive for temporary storage during processing
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("⚠️ Not running on Colab - some features may not work")

✅ Running on Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Install required packages
!pip install -q kagglehub insightface onnxruntime-gpu opencv-python-headless tqdm pillow

# Install pigz for parallel compression (much faster than gzip)
!apt-get install -y pigz

print("\n✅ Packages installed successfully")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pigz is already the newest version (2.6-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.

✅ Packages installed successfully


In [21]:
# Verify GPU is available
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name} ({gpu_memory:.1f} GB)")
else:
    print("⚠️ No GPU detected - preprocessing will be SLOW!")
    print("   Go to Runtime > Change runtime type > GPU")

✅ GPU Available: Tesla T4 (15.6 GB)


## 2. 🔐 Kaggle Authentication

You'll need a Kaggle account and API token:
1. Go to https://www.kaggle.com/settings
2. Click "Create New Token" under API section
3. Copy the token when prompted below

In [22]:
import kagglehub
import os

# Authenticate with Kaggle
# Method 1: Try to use Colab secret first (if in Colab)
if IN_COLAB:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get('KAGGLE_API_TOKEN')
        os.environ['KAGGLE_API_TOKEN'] = kaggle_token
        print("✅ Using Kaggle token from Colab secret")
    except Exception as e:
        print(f"⚠️ Colab secret not found: {e}")
        print("   Trying interactive login...")
        try:
            kagglehub.login()
        except Exception as e:
            print(f"❌ Authentication failed: {e}")
            print("   Please add 'KAGGLE_API_TOKEN' to Colab secrets or run kagglehub.login()")
            raise
else:
    # Method 2: Use environment variable or interactive login
    if 'KAGGLE_API_TOKEN' not in os.environ:
        print("⚠️ KAGGLE_API_TOKEN not in environment")
        print("   Trying interactive login...")
        kagglehub.login()
    else:
        print("✅ Using Kaggle token from environment variable")

print("\n✅ Kaggle authentication successful!")

✅ Using Kaggle token from Colab secret

✅ Kaggle authentication successful!


In [23]:
# Get your Kaggle username
# Note: kagglehub doesn't expose username directly, so we need to prompt for it
from getpass import getpass

print("📝 Enter your Kaggle username (from https://www.kaggle.com/settings)")
try:
    # Try to get from environment first
    KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', None)
    if not KAGGLE_USERNAME:
        # Prompt user
        KAGGLE_USERNAME = input("Kaggle username: ").strip()
        if not KAGGLE_USERNAME:
            raise ValueError("Username cannot be empty")
    print(f"✅ Using username: {KAGGLE_USERNAME}")
except Exception as e:
    print(f"❌ Failed to get username: {e}")
    raise

# Define dataset name
DATASET_NAME = "face-recognition-preprocessed"
default_handle = f"{KAGGLE_USERNAME}/{DATASET_NAME}"

print(f"\n📦 Default dataset handle: {default_handle}")
print("   (Press Enter to use default, or paste your specific handle)")
try:
    custom_handle = input("Dataset handle: ").strip()
except EOFError:
    custom_handle = ""

if custom_handle:
    DATASET_HANDLE = custom_handle
else:
    DATASET_HANDLE = default_handle

print(f"✅ Using dataset handle: {DATASET_HANDLE}")
print(f"   URL: https://www.kaggle.com/datasets/{DATASET_HANDLE}")


📝 Enter your Kaggle username (from https://www.kaggle.com/settings)
Kaggle username: mishafhasan
✅ Using username: mishafhasan

📦 Default dataset handle: mishafhasan/face-recognition-preprocessed
   (Press Enter to use default, or paste your specific handle)
Dataset handle: 
✅ Using dataset handle: mishafhasan/face-recognition-preprocessed
   URL: https://www.kaggle.com/datasets/mishafhasan/face-recognition-preprocessed


## 3. 📁 Directory Setup

In [24]:
import os
from pathlib import Path

# Define directories
BASE_DIR = Path("/content")
RAW_DATA_DIR = BASE_DIR / "raw_data"
PROCESSED_DIR = BASE_DIR / "processed_data"
TRAIN_DIR = PROCESSED_DIR / "train"
VAL_DIR = PROCESSED_DIR / "val"
OUTPUT_DIR = BASE_DIR / "output"  # For the final archive

# Create directories
for dir_path in [RAW_DATA_DIR, PROCESSED_DIR, TRAIN_DIR, VAL_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("📁 Directory structure created:")
print(f"   Raw data: {RAW_DATA_DIR}")
print(f"   Processed: {PROCESSED_DIR}")
print(f"   Output: {OUTPUT_DIR}")

📁 Directory structure created:
   Raw data: /content/raw_data
   Processed: /content/processed_data
   Output: /content/output


## 4. 📥 Dataset Download

We'll use the pre-processed VGGFace2-112x112 dataset from Kaggle:

### VGGFace2-112x112 Dataset:

| Property | Details |
|----------|---------|
| **Images** | ~3.3M+ face images |
| **Identities** | 9,131 unique identities |
| **Size** | Pre-processed, ~112x112 pixels |
| **Format** | JPG images organized by identity |
| **Preprocessing** | Already aligned and cropped to 112x112 |

### Why VGGFace2-112x112?

✅ **Pre-processed**: No need for face detection or alignment
✅ **Perfect Size**: Already 112x112 (MobileFaceNet input size)
✅ **Large Scale**: 9,131 identities for robust training
✅ **Research Standard**: VGGFace2 is a widely used benchmark
✅ **Ready to Use**: Can start training immediately

💡 **Kaggle Handle**: `yakhyokhuja/vggface2-112x112`

In [25]:
# Configuration for VGGFace2-112x112 dataset
DATASET_SOURCE = "vggface2-112x112"  # Pre-processed VGGFace2 (9.1K identities, 3.3M+ images)
DATASET_HANDLE = "yakhyokhuja/vggface2-112x112"

# For faster testing, you can limit the dataset
USE_SUBSET = False  # Set to True for testing with smaller subset
MAX_IDENTITIES = 500  # Only used if USE_SUBSET is True
MAX_IMAGES_PER_IDENTITY = 20  # Only used if USE_SUBSET is True

print(f"📊 Configuration:")
print(f"   Dataset source: {DATASET_SOURCE}")
print(f"   Dataset handle: {DATASET_HANDLE}")
print(f"   Use subset: {USE_SUBSET}")
if USE_SUBSET:
    print(f"   Max identities: {MAX_IDENTITIES}")
    print(f"   Max images per identity: {MAX_IMAGES_PER_IDENTITY}")

print(f"\n✅ VGGFace2-112x112 dataset will be downloaded")
print(f"   Expected: ~3.3M+ images, 9,131 identities")
print(f"   Images are pre-processed to 112x112 pixels")

📊 Configuration:
   Dataset source: vggface2-112x112
   Dataset handle: yakhyokhuja/vggface2-112x112
   Use subset: False

✅ VGGFace2-112x112 dataset will be downloaded
   Expected: ~3.3M+ images, 9,131 identities
   Images are pre-processed to 112x112 pixels


In [26]:
# Download VGGFace2-112x112 dataset from Kaggle
import kagglehub

print(f"\n{'='*60}")
print(f"📥 DOWNLOADING VGGFACE2-112x112 DATASET")
print(f"{'='*60}")
print(f"Dataset: {DATASET_HANDLE}")
print(f"This is a large dataset (~3.3M images), download may take 10-30 minutes")
print()

try:
    # Download latest version
    print("📥 Starting download...")
    path = kagglehub.dataset_download(DATASET_HANDLE)

    print(f"\n✅ Download successful!")
    print(f"   Path to dataset files: {path}")

    dataset_paths = {"vggface2": Path(path)}
    download_warnings = []

except Exception as e:
    print(f"❌ Download failed: {e}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Check internet connection")
    print(f"   2. Verify Kaggle authentication")
    print(f"   3. Check dataset availability: https://www.kaggle.com/datasets/{DATASET_HANDLE}")
    download_warnings = [f"VGGFace2-112x112 download failed: {e}"]
    dataset_paths = {}
    raise

print(f"\n{'='*60}")
print(f"📊 DOWNLOAD SUMMARY")
print(f"{'='*60}")
print(f"Successfully downloaded: {list(dataset_paths.keys())}")
if download_warnings:
    print(f"\n⚠️ Warnings:")
    for warning in download_warnings:
        print(f"   • {warning}")
print(f"{'='*60}")

if len(dataset_paths) == 0:
    raise RuntimeError("❌ Dataset download failed! Cannot continue.")


📥 DOWNLOADING VGGFACE2-112x112 DATASET
Dataset: yakhyokhuja/vggface2-112x112
This is a large dataset (~3.3M images), download may take 10-30 minutes

📥 Starting download...

✅ Download successful!
   Path to dataset files: /root/.cache/kagglehub/datasets/yakhyokhuja/vggface2-112x112/versions/1

📊 DOWNLOAD SUMMARY
Successfully downloaded: ['vggface2']


In [28]:
# Explore downloaded dataset structure with deep recursive listing
import traceback
from pathlib import Path

def explore_directory_structure(path, max_depth=3, current_depth=0):
    """Recursively explore directory structure to understand dataset organization."""
    path_obj = Path(path)
    results = {
        'dirs': [],
        'files': [],
        'structure_type': None,
        'has_archives': False,
        'identity_dirs': []
    }

    try:
        items = list(path_obj.iterdir())

        for item in sorted(items)[:50]:  # Limit to first 50 items for performance
            if item.is_dir():
                results['dirs'].append(item)
                # Check if this looks like an identity directory (starts with 'n' or 'id_')
                if (item.name.startswith('n') and item.name[1:].isdigit()) or item.name.startswith('id_'):
                    results['identity_dirs'].append(item)
            elif item.is_file():
                results['files'].append(item)
                if item.suffix.lower() in ['.tar', '.gz', '.zip', '.tgz']:
                    results['has_archives'] = True

    except Exception as e:
        print(f"   ⚠️ Error exploring {path}: {e}")

    return results

for name, path in dataset_paths.items():
    print(f"\n{'='*60}")
    print(f"📂 {name.upper()} DATASET STRUCTURE ANALYSIS")
    print(f"{'='*60}")
    print(f"Root path: {path}\n")

    try:
        # Level 0: Root directory
        root_info = explore_directory_structure(path)

        print(f"📊 Root Level Analysis:")
        print(f"   Directories: {len(root_info['dirs'])}")
        print(f"   Files: {len(root_info['files'])}")
        print(f"   Archives: {'Yes' if root_info['has_archives'] else 'No'}")
        print(f"   Identity folders in root: {len(root_info['identity_dirs'])}")

        if root_info['files']:
            print(f"\n   📄 Files in root:")
            for f in root_info['files'][:10]:
                size_mb = f.stat().st_size / (1024**2)
                print(f"      • {f.name}: {size_mb:.1f} MB")

        print(f"\n🔍 Structure Detection:")
        if root_info['has_archives']:
            print(f"   ⚠️ Dataset contains archives - may need extraction")

        if len(root_info['identity_dirs']) > 100:
            print(f"   ✅ Root contains identity folders directly ({len(root_info['identity_dirs'])} identities)")
        elif len(root_info['dirs']) > 0:
            print(f"   ✅ Root contains subdirectories - checking for train/test split...")

            for d in root_info['dirs'][:20]:
                try:
                    subdir_items = list(d.iterdir())
                    subdir_dirs = [x for x in subdir_items if x.is_dir()]
                    subdir_files = [x for x in subdir_items if x.is_file()]

                    print(f"      • {d.name}/:")
                    print(f"         - Contains: {len(subdir_dirs)} dirs, {len(subdir_files)} files")

                    if len(subdir_dirs) > 50:
                        print(f"         - Likely contains identity folders (train/test split)")
                        # Sample a few
                        for identity in subdir_dirs[:3]:
                            img_count = len([f for f in identity.iterdir() if f.is_file()])
                            print(f"            └─ {identity.name}: {img_count} images")
                except Exception as e:
                    print(f"      • {d.name}/: (unable to read: {e})")
        else:
            print(f"   ❌ No clear structure detected")

    except Exception as e:
        print(f"   ❌ Error analyzing structure: {e}")
        traceback.print_exc()


📂 VGGFACE2 DATASET STRUCTURE ANALYSIS
Root path: /root/.cache/kagglehub/datasets/yakhyokhuja/vggface2-112x112/versions/1

📊 Root Level Analysis:
   Directories: 1
   Files: 0
   Archives: No
   Identity folders in root: 0

🔍 Structure Detection:
   ✅ Root contains subdirectories - checking for train/test split...
      • vggface2_112x112/:
         - Contains: 8631 dirs, 0 files
         - Likely contains identity folders (train/test split)
            └─ id_4661: 505 images
            └─ id_6077: 416 images
            └─ id_8023: 204 images


In [30]:
# Verify VGGFace2-112x112 dataset structure
# This dataset is already organized with identities in folders

def verify_vggface2_structure(vggface2_path):
    """
    Intelligently detect VGGFace2 dataset structure with multiple fallback strategies.
    """
    vggface2_path = Path(vggface2_path)

    print(f"\n{'='*60}")
    print(f"🔍 INTELLIGENT DATASET STRUCTURE DETECTION")
    print(f"{'='*60}")
    print(f"Scanning: {vggface2_path}\n")

    train_dir = None
    test_dir = None

    # Strategy 1: Look for train/test in root
    print(f"⌆ Strategy 1: Checking root for train/test directories...")
    try:
        for item in vggface2_path.iterdir():
            if not item.is_dir():
                continue

            name_lower = item.name.lower()
            if 'train' in name_lower and not train_dir:
                subdirs = [d for d in item.iterdir() if d.is_dir()]
                if len(subdirs) > 10:
                    train_dir = item
                    print(f"   ✅ Found training directory: {item.name}/ ({len(subdirs)} identities)")
            elif ('test' in name_lower or 'val' in name_lower) and not test_dir:
                subdirs = [d for d in item.iterdir() if d.is_dir()]
                if len(subdirs) > 0:
                    test_dir = item
                    print(f"   ✅ Found test directory: {item.name}/ ({len(subdirs)} identities)")
    except Exception as e:
        print(f"   ☐ Strategy 1 failed: {e}")

    # Strategy 2: Check nested directories
    if not train_dir:
        print(f"\n⌆ Strategy 2: Checking nested directories...")
        try:
            for item in vggface2_path.iterdir():
                if not item.is_dir(): continue
                subdirs = [d for d in item.iterdir() if d.is_dir()]
                if len(subdirs) > 100:
                    train_dir = item
                    print(f"   ✅ Found identities in: {item.name}/ ({len(subdirs)} identities)")
                    break
        except Exception as e:
            print(f"   ☐ Strategy 2 failed: {e}")

    return train_dir, test_dir

# Run the verification
organized_dirs = {}
print(f"🔍 DATASET VERIFICATION")
for name, path in dataset_paths.items():
    t_dir, v_dir = verify_vggface2_structure(path)
    organized_dirs[name] = {"train": t_dir, "test": v_dir}

    if t_dir:
        train_identities = [d for d in t_dir.iterdir() if d.is_dir()]
        print(f"\n✅ Training data located")
        print(f"   Path: {t_dir}")
        print(f"   Identities: {len(train_identities)}")

🔍 DATASET VERIFICATION

🔍 INTELLIGENT DATASET STRUCTURE DETECTION
Scanning: /root/.cache/kagglehub/datasets/yakhyokhuja/vggface2-112x112/versions/1

⌆ Strategy 1: Checking root for train/test directories...

⌆ Strategy 2: Checking nested directories...
   ✅ Found identities in: vggface2_112x112/ (8631 identities)

✅ Training data located
   Path: /root/.cache/kagglehub/datasets/yakhyokhuja/vggface2-112x112/versions/1/vggface2_112x112
   Identities: 8631


## 5. ✅ Dataset Ready - No Preprocessing Needed!

✨ **Good News**: VGGFace2-112x112 is already pre-processed!

All images are:
- ✅ Already aligned
- ✅ Already cropped to faces
- ✅ Already resized to 112x112
- ✅ Ready for training

**No face detection, alignment, or resizing is needed!**

In [32]:
# ========================================
# VALIDATION CHECKPOINT
# ========================================
# Verify dataset was found before continuing

print("\n" + "="*70)
print("🔍 DATASET VALIDATION CHECKPOINT")
print("="*70)

dataset_valid = False
dataset_info = {}

if 'vggface2' in organized_dirs:
    train_dir = organized_dirs['vggface2']['train']
    test_dir = organized_dirs['vggface2']['test']

    if train_dir and train_dir.exists():
        try:
            identity_count = len([d for d in train_dir.iterdir() if d.is_dir()])

            if identity_count > 0:
                dataset_valid = True
                dataset_info['train_dir'] = train_dir
                dataset_info['train_identities'] = identity_count
                dataset_info['test_dir'] = test_dir

                print(f"✅ Dataset validation PASSED")
                print(f"\n📊 Dataset Information:")
                print(f"   Training directory: {train_dir}")
                print(f"   Training identities: {identity_count}")
                if test_dir and test_dir.exists():
                    test_count = len([d for d in test_dir.iterdir() if d.is_dir()])
                    print(f"   Test directory: {test_dir}")
                    print(f"   Test identities: {test_count}")
                    dataset_info['test_identities'] = test_count
            else:
                print(f"❌ Training directory is empty!")
                print(f"   Path: {train_dir}")
        except Exception as e:
            print(f"❌ Error validating dataset: {e}")
    else:
        print(f"❌ Training directory not found or doesn't exist")
        if train_dir:
            print(f"   Path tried: {train_dir}")
else:
    print(f"❌ Dataset not found in organized_dirs")
    print(f"   Available keys: {list(organized_dirs.keys())}")

if not dataset_valid:
    print(f"\n{'='*70}")
    print(f"⚠️ CANNOT CONTINUE - DATASET NOT AVAILABLE")
    print(f"{'='*70}")
    print(f"\n💡 Possible solutions:")
    print(f"   1. Check dataset downloaded successfully (Cell 12)")
    print(f"   2. Verify dataset structure detection (Cell 14)")
    print(f"   3. Try different Kaggle dataset if structure incompatible")
    print(f"   4. Manually inspect downloaded path and adjust code")
    print(f"\n🛑 Please fix dataset issues before running subsequent cells")
else:
    print(f"\n{'='*70}")
    print(f"✅ VALIDATION PASSED - Ready to proceed with processing")
    print(f"{'='*70}")



🔍 DATASET VALIDATION CHECKPOINT
✅ Dataset validation PASSED

📊 Dataset Information:
   Training directory: /root/.cache/kagglehub/datasets/yakhyokhuja/vggface2-112x112/versions/1/vggface2_112x112
   Training identities: 8631

✅ VALIDATION PASSED - Ready to proceed with processing


In [ ]:
# Verify sample images from the dataset
from PIL import Image
import random

print("📸 Verifying sample images...")

try:
    # Get a sample identity from train directory
    if 'vggface2' in organized_dirs and organized_dirs['vggface2']['train']:
        train_dir = organized_dirs['vggface2']['train']
        identity_folders = [d for d in train_dir.iterdir() if d.is_dir()]

        if identity_folders:
            # Pick a random identity
            sample_identity = random.choice(identity_folders)
            print(f"   Sample identity: {sample_identity.name}")

            # Get images from this identity
            image_files = [f for f in sample_identity.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]

            if image_files:
                # Check first image
                sample_img_path = image_files[0]
                img = Image.open(sample_img_path)
                print(f"   Sample image: {sample_img_path.name}")
                print(f"   Size: {img.size}")
                print(f"   Mode: {img.mode}")

                if img.size == (112, 112):
                    print(f"   ✅ Images are correctly sized (112x112)!")
                else:
                    print(f"   ⚠️ Warning: Image size is {img.size}, expected (112, 112)")

                print(f"   Total images in this identity: {len(image_files)}")
            else:
                print(f"   ⚠️ No images found in {sample_identity.name}")
        else:
            print(f"   ⚠️ No identity folders found")
    else:
        print(f"   ⚠️ Could not access training directory")

except Exception as e:
    print(f"   ❌ Error verifying images: {e}")

print("\n✅ Dataset verification complete!")


📸 Verifying sample images...
   ⚠️ Could not access training directory

✅ Dataset verification complete!


In [ ]:
# Copy dataset to processing directory (if needed for train/val split)
import shutil
from tqdm.auto import tqdm

print("📂 Preparing dataset for training...")

# Check if we need to create train/val split or if it already exists
if 'vggface2' in organized_dirs:
    source_train = organized_dirs['vggface2']['train']
    source_test = organized_dirs['vggface2']['test']

    print(f"\n   Source train directory: {source_train}")
    if source_test:
        print(f"   Source test directory: {source_test}")

    # Count identities
    if source_train:
        train_identities = [d for d in source_train.iterdir() if d.is_dir()]
        print(f"   Training identities: {len(train_identities)}")

        # If using subset, limit identities
        if USE_SUBSET:
            print(f"\n   ⚠️ Using subset mode:")
            print(f"      Limiting to {MAX_IDENTITIES} identities")
            train_identities = train_identities[:MAX_IDENTITIES]

    if source_test:
        test_identities = [d for d in source_test.iterdir() if d.is_dir()]
        print(f"   Test identities: {len(test_identities)}")
        if USE_SUBSET:
            test_identities = test_identities[:MAX_IDENTITIES]

else:
    print("   ⚠️ No dataset found in organized_dirs")

print("\n✅ Dataset ready - images are pre-processed to 112x112!")


📂 Preparing dataset for training...
   ⚠️ No dataset found in organized_dirs

✅ Dataset ready - images are pre-processed to 112x112!


In [36]:
# Copy dataset to working directory (for organization and splitting)
def copy_dataset_subset(source_dir, output_dir, max_identities=None):
    """
    Copy a subset of the dataset for processing.
    Since images are already preprocessed, we just copy them.
    """
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"📂 Copying dataset from {source_dir.name}...")

    # Get identity directories
    identity_dirs = [d for d in source_dir.iterdir() if d.is_dir()]

    if max_identities:
        identity_dirs = identity_dirs[:max_identities]

    print(f"   Processing {len(identity_dirs)} identities...")

    stats = {"identities": 0, "images": 0, "skipped": 0}

    for identity_dir in tqdm(identity_dirs, desc="Copying identities"):
        identity_name = identity_dir.name
        identity_output = output_dir / identity_name
        identity_output.mkdir(parents=True, exist_ok=True)

        # Get image files
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        images = [f for f in identity_dir.iterdir()
                  if f.is_file() and f.suffix.lower() in image_extensions]

        if len(images) == 0:
            stats["skipped"] += 1
            continue

        # Copy images
        copied = 0
        for img_path in images:
            try:
                output_path = identity_output / img_path.name
                if not output_path.exists():
                    shutil.copy2(img_path, output_path)
                copied += 1
                stats["images"] += 1
            except Exception as e:
                stats["skipped"] += 1

        if copied > 0:
            stats["identities"] += 1

    print(f"\n✅ Copy complete!")
    print(f"   Identities: {stats['identities']}")
    print(f"   Images: {stats['images']}")
    print(f"   Skipped: {stats['skipped']}")

    return stats

In [ ]:
# Process VGGFace2 dataset
if 'vggface2' in organized_dirs:
    train_src = organized_dirs['vggface2']['train']

    if train_src and train_src.exists():
        print("\n" + "="*70)
        print("🔄 PROCESSING VGGFACE2-112x112 DATASET")
        print("="*70)

        # Copy training data (use max_identities to control subset size if needed)
        stats = copy_dataset_subset(
            source_dir=train_src,
            output_dir=TRAIN_DIR,  # Fixed: was output_base / "vggface2_train"
            max_identities=None  # Set to a number (e.g., 1000) to limit dataset size
        )

        print(f"\n✨ Dataset processing complete!")
        all_stats = stats
    else:
        print("⚠️ Could not access training directory")
        all_stats = {"identities": 0, "images": 0, "skipped": 0}  # Default if processing fails
else:
    print("⚠️ Dataset not found in organized_dirs")
    all_stats = {"identities": 0, "images": 0, "skipped": 0}  # Default if dataset not found



⚠️ Dataset not found in organized_dirs


In [38]:
# Split into train/val
import random
import shutil
from tqdm.auto import tqdm

VAL_SPLIT = 0.15  # 15% for validation
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

print("📊 Creating train/val split...")
print(f"   Random seed: {RANDOM_SEED} (for reproducibility)")

# Check if TRAIN_DIR exists and has data
if not TRAIN_DIR.exists():
    print(f"❌ Error: TRAIN_DIR does not exist: {TRAIN_DIR}")
    print(f"   Cannot proceed with train/val split")
    val_count = 0
    train_count = 0
else:
    identity_dirs = [d for d in TRAIN_DIR.iterdir() if d.is_dir()]
    print(f"   Total identities: {len(identity_dirs)}")

    if len(identity_dirs) == 0:
        print(f"⚠️ Warning: No identity directories found in TRAIN_DIR")
        print(f"   Path: {TRAIN_DIR}")
        print(f"   This means no data was copied in the previous step")
        val_count = 0
        train_count = 0
    else:
        # For each identity, move some images to validation
        val_count = 0
        train_count = 0

        for identity_dir in tqdm(identity_dirs, desc="Splitting"):
            images = list(identity_dir.glob("*"))

            if len(images) < 2:
                train_count += len(images)
                continue

            # Calculate number for validation
            n_val = max(1, int(len(images) * VAL_SPLIT))

            # Randomly select validation images
            random.shuffle(images)
            val_images = images[:n_val]

            # Move to validation directory
            val_identity_dir = VAL_DIR / identity_dir.name
            val_identity_dir.mkdir(parents=True, exist_ok=True)

            for img in val_images:
                shutil.move(str(img), str(val_identity_dir / img.name))
                val_count += 1

            train_count += len(images) - n_val

        print(f"\n✅ Split complete!")
        print(f"   Validation images: {val_count}")
        print(f"   Train images: {train_count}")


📊 Creating train/val split...
   Random seed: 42 (for reproducibility)
   Total identities: 0
⚠️ Warning: No identity directories found in TRAIN_DIR
   Path: /content/processed_data/train
   This means no data was copied in the previous step


## 5.5 🧹 Storage Cleanup

Now that we've copied and organized the data, we'll delete the original downloaded dataset to save storage space before compression.

In [ ]:
# Clean up original downloaded dataset to save storage
import shutil

print("🧹 Cleaning up original dataset to save storage...")

# Only delete if we successfully processed the data
if train_count > 0 or val_count > 0:
    try:
        # Get the original dataset path
        if 'vggface2' in dataset_paths:
            original_path = dataset_paths['vggface2']
            
            # Calculate size before deletion
            if original_path.exists():
                original_size_gb = get_dir_size(str(original_path)) / (1024**3)
                
                print(f"   Original dataset path: {original_path}")
                print(f"   Original dataset size: {original_size_gb:.2f} GB")
                print(f"   Deleting to free up storage...")
                
                # Delete the original dataset
                shutil.rmtree(original_path)
                
                print(f"\n✅ Original dataset deleted!")
                print(f"   Storage freed: {original_size_gb:.2f} GB")
                print(f"   Processed data retained in: {PROCESSED_DIR}")
            else:
                print(f"   ⚠️ Original dataset path not found, may already be deleted")
        else:
            print(f"   ⚠️ No dataset path found in dataset_paths")
            
    except Exception as e:
        print(f"   ⚠️ Error cleaning up dataset: {e}")
        print(f"   This is non-critical - you can manually delete it later")
else:
    print(f"   ⚠️ Skipping cleanup - no data was processed")
    print(f"   Original dataset retained for debugging")

print(f"\n💾 Current storage usage:")
print(f"   Processed data: {total_size:.2f} GB")
print(f"   (Original dataset removed to save space)")


## 6. 📦 Compression with pigz (Parallel gzip)

**pigz** uses all CPU cores for compression, making it 4-8x faster than standard gzip.

In [39]:
# Check directory sizes before compression
def get_dir_size(path):
    """Get directory size in bytes."""
    total = 0
    for entry in os.scandir(path):
        if entry.is_file():
            total += entry.stat().st_size
        elif entry.is_dir():
            total += get_dir_size(entry.path)
    return total

train_size = get_dir_size(TRAIN_DIR) / (1024**3)  # GB
val_size = get_dir_size(VAL_DIR) / (1024**3)  # GB
total_size = train_size + val_size

print(f"📊 Dataset sizes:")
print(f"   Train: {train_size:.2f} GB")
print(f"   Val: {val_size:.2f} GB")
print(f"   Total: {total_size:.2f} GB")
print(f"\n📦 Estimated compressed size: {total_size * 0.7:.2f} GB")

📊 Dataset sizes:
   Train: 0.00 GB
   Val: 0.00 GB
   Total: 0.00 GB

📦 Estimated compressed size: 0.00 GB


In [40]:
# Create metadata file
import json
from datetime import datetime

# Count actual images
train_images = sum(1 for _ in TRAIN_DIR.rglob("*.jpg")) + sum(1 for _ in TRAIN_DIR.rglob("*.png"))
val_images = sum(1 for _ in VAL_DIR.rglob("*.jpg")) + sum(1 for _ in VAL_DIR.rglob("*.png"))
train_ids = len([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
val_ids = len([d for d in VAL_DIR.iterdir() if d.is_dir()])

metadata = {
    "name": "Face Recognition Preprocessed Dataset",
    "description": "Pre-aligned VGGFace2 112x112 face images for face recognition training",
    "created": datetime.now().isoformat(),
    "source_datasets": list(dataset_paths.keys()),
    "preprocessing": {
        "alignment": "Pre-aligned by VGGFace2 team",
        "face_detector": "MTCNN (by VGGFace2 team)",
        "output_size": "112x112",
        "format": "RGB",
        "note": "Images already aligned and cropped, no additional preprocessing applied"
    },
    "statistics": {
        "train_images": train_images,
        "train_identities": train_ids,
        "val_images": val_images,
        "val_identities": val_ids,
        "total_images": train_images + val_images
    },
    "structure": {
        "train/": "Training images organized by identity",
        "val/": "Validation images organized by identity"
    }
}

# Save metadata
metadata_path = PROCESSED_DIR / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata saved:")
print(json.dumps(metadata, indent=2))

✅ Metadata saved:
{
  "name": "Face Recognition Preprocessed Dataset",
  "description": "Pre-aligned VGGFace2 112x112 face images for face recognition training",
  "created": "2026-02-12T13:42:47.604778",
  "source_datasets": [
    "vggface2"
  ],
  "preprocessing": {
    "alignment": "Pre-aligned by VGGFace2 team",
    "face_detector": "MTCNN (by VGGFace2 team)",
    "output_size": "112x112",
    "format": "RGB",
    "note": "Images already aligned and cropped, no additional preprocessing applied"
  },
  "statistics": {
    "train_images": 0,
    "train_identities": 0,
    "val_images": 0,
    "val_identities": 0,
    "total_images": 0
  },
  "structure": {
    "train/": "Training images organized by identity",
    "val/": "Validation images organized by identity"
  }
}


In [41]:
# Compress with pigz (parallel gzip) or fallback to standard gzip
import subprocess
import shutil
import time

# Validate we have data to compress
if total_size == 0:
    print("⚠️ Warning: No data to compress (total_size = 0)")
    print("   Skipping compression step")
    print("\n💡 This usually means:")
    print("   • Dataset was not successfully copied to TRAIN_DIR/VAL_DIR")
    print("   • Dataset validation failed in earlier steps")
    print("   • Please check previous cells for errors")
else:
    ARCHIVE_NAME = "face_recognition_dataset.tar.gz"
    ARCHIVE_PATH = OUTPUT_DIR / ARCHIVE_NAME

    print(f"📦 Compressing dataset...")
    print(f"   Total size to compress: {total_size:.2f} GB")

    start_time = time.time()

    # Check if pigz is available
    pigz_available = shutil.which('pigz') is not None

    if pigz_available:
        print(f"   Using pigz (parallel gzip) for faster compression")
        try:
            # Create tar and compress with pigz using separate processes (safer than shell=True)
            import tempfile

            # First create tar archive
            print(f"\n⏳ Creating tar archive...")
            tar_temp = tempfile.NamedTemporaryFile(delete=False, suffix='.tar')
            tar_temp.close()

            # Create tar without compression
            tar_result = subprocess.run(
                ['tar', '-cf', tar_temp.name, '-C', str(PROCESSED_DIR), '.'],
                capture_output=True,
                text=True
            )

            if tar_result.returncode != 0:
                raise Exception(f"Tar failed: {tar_result.stderr}")

            print(f"   Compressing with pigz...")
            # Compress with pigz (omit -p flag to let pigz auto-detect cores)
            with open(tar_temp.name, 'rb') as tar_file:
                with open(ARCHIVE_PATH, 'wb') as gz_file:
                    pigz_result = subprocess.run(
                        ['pigz', '-6'],  # Removed -p 0, pigz auto-detects cores by default
                        stdin=tar_file,
                        stdout=gz_file,
                        stderr=subprocess.PIPE
                    )

            # Clean up temp file
            os.unlink(tar_temp.name)

            if pigz_result.returncode != 0:
                raise Exception(f"Pigz failed: {pigz_result.stderr.decode()}")

        except Exception as e:
            print(f"   ⚠️ Pigz compression failed: {e}")
            print(f"   Falling back to standard gzip...")
            pigz_available = False

    if not pigz_available:
        print(f"   Using standard gzip compression (slower)")
        print(f"\n⏳ Creating compressed tar archive...")

        # Use Python's tarfile module (cross-platform, no shell=True)
        import tarfile

        with tarfile.open(ARCHIVE_PATH, 'w:gz', compresslevel=6) as tar:
            tar.add(PROCESSED_DIR, arcname='.')

    elapsed = time.time() - start_time

    if ARCHIVE_PATH.exists():
        archive_size = ARCHIVE_PATH.stat().st_size / (1024**3)

        # Safe division with zero check
        if total_size > 0:
            compression_ratio = (total_size - archive_size) / total_size * 100
        else:
            compression_ratio = 0

        print(f"\n✅ Compression complete!")
        print(f"   Time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
        print(f"   Original size: {total_size:.2f} GB")
        print(f"   Compressed size: {archive_size:.2f} GB")
        print(f"   Compression ratio: {compression_ratio:.1f}%")
        print(f"   Archive: {ARCHIVE_PATH}")
    else:
        print(f"\n❌ Compression failed - archive not created")


⚠️ Warning: No data to compress (total_size = 0)
   Skipping compression step

💡 This usually means:
   • Dataset was not successfully copied to TRAIN_DIR/VAL_DIR
   • Dataset validation failed in earlier steps
   • Please check previous cells for errors


## 7. 📤 Prepare Files for Manual Kaggle Upload

Files are ready for manual upload. Follow the instructions below to upload to Kaggle.

In [42]:
# Prepare upload directory with archive and metadata
UPLOAD_DIR = BASE_DIR / "kaggle_upload"
UPLOAD_DIR.mkdir(exist_ok=True)

# Copy archive and metadata to upload directory
shutil.copy(ARCHIVE_PATH, UPLOAD_DIR / ARCHIVE_NAME)
shutil.copy(metadata_path, UPLOAD_DIR / "metadata.json")

# Create README for manual upload
readme_content = f"""# Face Recognition Dataset - VGGFace2 112x112

VGGFace2 dataset preprocessed and ready for training face recognition models.

## 📦 Contents

- `{ARCHIVE_NAME}` - Compressed dataset (tar.gz)
- `metadata.json` - Dataset metadata and statistics

## 📊 Statistics

- **Training Images**: {train_images:,}
- **Training Identities**: {train_ids:,}
- **Validation Images**: {val_images:,}
- **Validation Identities**: {val_ids:,}
- **Total Images**: {train_images + val_images:,}

## 🔧 Dataset Details

- **Source**: VGGFace2 (yakhyokhuja/vggface2-112x112)
- **Image Size**: 112x112 pixels (pre-processed)
- **Format**: RGB JPG images
- **Alignment**: Pre-aligned faces
- **Ready to train**: No additional preprocessing needed

## 🚀 Usage in Training Notebook

### Step 1: Download the dataset

```python
import kagglehub

# Download dataset (cached after first download)
path = kagglehub.dataset_download("yakhyokhuja/vggface2-112x112")
print(f"Dataset downloaded to: {{path}}")
```

### Step 2: Access the data directly

```python
from pathlib import Path

# Dataset is organized and ready to use
TRAIN_DIR = Path(path) / "train"
TEST_DIR = Path(path) / "test"

print(f"Training data: {{TRAIN_DIR}}")
print(f"Test data: {{TEST_DIR}}")
```

### Step 3: Load with PyTorch

```python
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = ImageFolder(TRAIN_DIR, transform=transform)
print(f"Training samples: {{len(train_dataset)}}")
```

## 📁 Dataset Structure

```
data/
├── train/
│   ├── n000001/
│   │   ├── 0001_01.jpg
│   │   ├── 0002_01.jpg
│   │   └── ...
│   ├── n000002/
│   └── ...
└── test/
    ├── n000001/
    └── ...
```

## 📅 Processed

{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 📝 Notes

- All images are pre-aligned and cropped to 112x112
- Source: VGGFace2 dataset (yakhyokhuja/vggface2-112x112)
- Ready for training face recognition models
- Compatible with PyTorch, TensorFlow, and other frameworks
- No additional preprocessing required
"""

with open(UPLOAD_DIR / "README.md", 'w') as f:
    f.write(readme_content)

print("✅ Upload directory prepared:")
for f in UPLOAD_DIR.iterdir():
    size = f.stat().st_size / (1024**2)  # MB
    print(f"   {f.name}: {size:.1f} MB")

✅ Upload directory prepared:
   README.md: 0.0 MB
   face_recognition_dataset.tar.gz: 0.0 MB
   metadata.json: 0.0 MB


In [43]:
# Prepare files for manual upload to Kaggle
print(f"📦 Dataset prepared for manual upload!")
print(f"\n📂 Upload directory: {UPLOAD_DIR}")
print(f"   Files ready:")

total_size_mb = 0
for f in UPLOAD_DIR.iterdir():
    size = f.stat().st_size / (1024**2)  # MB
    total_size_mb += size
    print(f"   ✓ {f.name}: {size:.1f} MB")

print(f"\n   Total size: {total_size_mb:.1f} MB ({total_size_mb/1024:.2f} GB)")

print(f"\n" + "="*70)
print(f"📝 MANUAL UPLOAD INSTRUCTIONS")
print(f"="*70)

print(f"""
🌐 Step 1: Go to Kaggle
   → Open: https://www.kaggle.com/datasets

📤 Step 2: Create New Dataset
   → Click "New Dataset" button in top right

📁 Step 3: Upload Files
   → Click "Select Files to Upload"
   → Navigate to: {UPLOAD_DIR}
   → Select ALL files:
      • {ARCHIVE_NAME}
      • metadata.json
      • README.md
   → Click "Open" to upload

⚙️ Step 4: Configure Dataset
   → Title: Face Recognition Preprocessed Dataset
   → Subtitle: Aligned face images for training
   → Dataset slug: {DATASET_NAME}
   → Public/Private: Choose "Private" (recommended initially)

📝 Step 5: Add Description
   → The README.md content will be used automatically
   → Or copy-paste from the README.md file

🏷️ Step 6: Add Tags (Optional)
   → face-recognition
   → computer-vision
   → deep-learning
   → preprocessing

✅ Step 7: Create Dataset
   → Click "Create" button
   → Wait for upload to complete (may take 10-30 minutes)

🔗 Step 8: Get Dataset URL
   → Your dataset will be at:
   → https://www.kaggle.com/datasets/{DATASET_HANDLE}

💡 Alternative: Download to Local Machine
   If running in Colab, download files to your computer first:
""")

print(f"""
# Run in Colab to download files to your computer:
from google.colab import files

# Download compressed dataset
print("📥 Downloading {ARCHIVE_NAME}...")
files.download(str(UPLOAD_DIR / "{ARCHIVE_NAME}"))

# Download metadata
print("📥 Downloading metadata.json...")
files.download(str(UPLOAD_DIR / "metadata.json"))

# Download README
print("📥 Downloading README.md...")
files.download(str(UPLOAD_DIR / "README.md"))

print("✅ All files downloaded! Now upload to Kaggle manually.")
""")

print(f"\n" + "="*70)
print(f"📋 QUICK REFERENCE")
print(f"="*70)
print(f"Dataset Handle: {DATASET_HANDLE}")
print(f"Upload Directory: {UPLOAD_DIR}")
print(f"Total Size: {total_size_mb/1024:.2f} GB")
print(f"\n✨ After upload, use in training notebook:")
print(f"   kagglehub.dataset_download('{DATASET_HANDLE}')")
print(f"="*70)

📦 Dataset prepared for manual upload!

📂 Upload directory: /content/kaggle_upload
   Files ready:
   ✓ README.md: 0.0 MB
   ✓ face_recognition_dataset.tar.gz: 0.0 MB
   ✓ metadata.json: 0.0 MB

   Total size: 0.0 MB (0.00 GB)

📝 MANUAL UPLOAD INSTRUCTIONS

🌐 Step 1: Go to Kaggle
   → Open: https://www.kaggle.com/datasets

📤 Step 2: Create New Dataset
   → Click "New Dataset" button in top right

📁 Step 3: Upload Files
   → Click "Select Files to Upload"
   → Navigate to: /content/kaggle_upload
   → Select ALL files:
      • face_recognition_dataset.tar.gz
      • metadata.json
      • README.md
   → Click "Open" to upload

⚙️ Step 4: Configure Dataset
   → Title: Face Recognition Preprocessed Dataset
   → Subtitle: Aligned face images for training
   → Dataset slug: face-recognition-preprocessed
   → Public/Private: Choose "Private" (recommended initially)

📝 Step 5: Add Description
   → The README.md content will be used automatically
   → Or copy-paste from the README.md file

🏷️ Ste

## 8. ✅ (Optional) Test Download After Upload

After you've manually uploaded the dataset to Kaggle, you can test downloading it here.

In [ ]:
# ============================================================
# TEST DATASET DOWNLOAD (Run AFTER uploading to Kaggle)
# ============================================================
#
# This cell verifies that your uploaded dataset is accessible

import kagglehub
from pathlib import Path
import os

print("📥 Testing dataset download...")
print(f"⚠️  Make sure you've completed these steps first:")
print(f"   1. Dataset uploaded to Kaggle")
print(f"   2. Dataset privacy set to 'Public' (Settings → Privacy)")
print(f"   3. Kaggle API credentials configured")
print(f"\n   Expected dataset: {DATASET_HANDLE}")
print()


print("\n" + "="*60)
print("🔄 Attempting download...")
print("="*60 + "\n")

try:
    download_path = kagglehub.dataset_download(DATASET_HANDLE)
    print(f"✅ Download successful!")
    print(f"   Path: {download_path}")

    # List contents
    print(f"\n📂 Contents:")
    download_dir = Path(download_path)
    total_size = 0
    file_count = 0

    for f in sorted(download_dir.iterdir()):
        if f.is_file():
            size_mb = f.stat().st_size / (1024**2)  # MB
            total_size += size_mb
            file_count += 1
            print(f"   📄 {f.name}: {size_mb:.1f} MB")
        elif f.is_dir():
            # Count files in subdirectory
            try:
                subfiles = list(f.glob('**/*'))
                # Filter to get only actual files (not directories)
                file_list = [x for x in subfiles if x.is_file()]
                subfile_count = len(file_list)
                
                # Calculate total size of files in this subdirectory
                subdir_size_mb = 0
                for file_path in file_list:
                    try:
                        subdir_size_mb += file_path.stat().st_size / (1024**2)
                    except:
                        pass  # Skip files that can't be accessed
                
                # Add to totals
                file_count += subfile_count
                total_size += subdir_size_mb
                
                print(f"   📁 {f.name}/: {subfile_count} files ({subdir_size_mb:.1f} MB)")
            except Exception as e:
                print(f"   📁 {f.name}/ (unable to analyze: {e})")

    print(f"\n📊 Summary:")
    print(f"   Total files: {file_count:,}")
    print(f"   Total size: {total_size:.1f} MB ({total_size/1024:.2f} GB)")

    print(f"\n✅ Dataset is accessible from Kaggle!")
    print(f"\n🎯 Use this code in training notebooks:")
    print(f"   import kagglehub")
    print(f"   path = kagglehub.dataset_download('{DATASET_HANDLE}')")
    print(f"   print('Dataset path:', path)")

except Exception as e:
    error_msg = str(e)
    print(f"❌ Download failed: {error_msg}")
    print(f"\n💡 Troubleshooting Guide:")
    print(f"\n1️⃣  DATASET PRIVACY (Most Common Issue)")
    print(f"   • Go to: https://www.kaggle.com/datasets/{DATASET_HANDLE}")
    print(f"   • Click 'Settings' tab")
    print(f"   • Under 'Privacy', change from 'Private' → 'Public'")
    print(f"   • Click 'Save'")

    print(f"\n2️⃣  AUTHENTICATION")
    if "403" in error_msg or "permission" in error_msg.lower():
        print(f"   ⚠️  403 Permission Error detected")
        print(f"   • This usually means the dataset is PRIVATE")
        print(f"   • Even if you own it, you must make it PUBLIC to access via API")

    if not has_api_token and not has_kaggle_json:
        print(f"   • Kaggle credentials not configured")
        print(f"   • Run: kagglehub.login()")
        print(f"   • Or set up ~/.kaggle/kaggle.json")

    print(f"\n3️⃣  DATASET HANDLE")
    print(f"   • Current handle: {DATASET_HANDLE}")
    print(f"   • Format should be: username/dataset-name")
    print(f"   • Check for typos in username or dataset name")

    print(f"\n4️⃣  DATASET UPLOAD")
    print(f"   • Verify upload completed successfully")
    print(f"   • Check dataset page loads: https://www.kaggle.com/datasets/{DATASET_HANDLE}")

    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"\n   ⚠️  404 Not Found Error detected")
        print(f"   • Dataset doesn't exist at this URL")
        print(f"   • Double-check the dataset handle")
        print(f"   • Verify upload completed")

    print(f"\n" + "="*60)
    print(f"🔍 Quick Checklist:")
    print(f"="*60)
    print(f"   [ ] Dataset uploaded to Kaggle")
    print(f"   [ ] Dataset is PUBLIC (not private)")
    print(f"   [ ] Kaggle API credentials configured")
    print(f"   [ ] Dataset handle is correct: {DATASET_HANDLE}")
    print(f"   [ ] Can access manually: https://www.kaggle.com/datasets/{DATASET_HANDLE}")



📥 Testing dataset download...
⚠️  Make sure you've completed these steps first:
   1. Dataset uploaded to Kaggle
   2. Dataset privacy set to 'Public' (Settings → Privacy)
   3. Kaggle API credentials configured

   Expected dataset: yakhyokhuja/vggface2-112x112


🔄 Attempting download...

✅ Download successful!
   Path: /root/.cache/kagglehub/datasets/yakhyokhuja/vggface2-112x112/versions/1

📂 Contents:
   📁 vggface2_112x112/: 3137807 files

📊 Summary:
   Total files: 0
   Total size: 0.0 MB

✅ Dataset is accessible from Kaggle!

🎯 Use this code in training notebooks:
   import kagglehub
   path = kagglehub.dataset_download('yakhyokhuja/vggface2-112x112')
   print('Dataset path:', path)


## 9. 📋 Usage Instructions for Training Notebook

Copy this code to your training notebook to use the dataset:

In [45]:
# Print usage instructions
usage_code = """# ============================================
# Add this to your training notebook
# ============================================

import kagglehub
from pathlib import Path

# Authenticate with Kaggle (if needed)
# kagglehub.login()

# Download VGGFace2-112x112 dataset (cached after first download!)
print("📥 Downloading VGGFace2-112x112 dataset...")
dataset_path = kagglehub.dataset_download("yakhyokhuja/vggface2-112x112")
print(f"✅ Downloaded to: {dataset_path}")

# Access the data directly - no extraction needed!
DATA_DIR = Path(dataset_path)
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

# Verify structure
if TRAIN_DIR.exists():
    train_ids = len([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
    print(f"\\n✅ Dataset ready!")
    print(f"   Train identities: {train_ids}")
    print(f"   Train path: {TRAIN_DIR}")

if TEST_DIR.exists():
    test_ids = len([d for d in TEST_DIR.iterdir() if d.is_dir()])
    print(f"   Test identities: {test_ids}")
    print(f"   Test path: {TEST_DIR}")

print(f"\\n✨ Images are pre-processed to 112x112 - ready to train!")

# Example: Load with PyTorch
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = ImageFolder(TRAIN_DIR, transform=transform)
print(f"\\nTraining samples: {len(train_dataset)}")
"""

print(usage_code)
print("\n" + "="*60)
print("Copy the code above to your training notebook!")
print("="*60)
print("\n💡 Benefits:")
print("   ✅ No preprocessing needed - images are 112x112")
print("   ✅ Fast downloads with kagglehub caching")
print("   ✅ Direct access - no extraction required")
print("   ✅ Large-scale dataset with 9K+ identities")

# ============================================
# Add this to your training notebook
# ============================================

import kagglehub
from pathlib import Path

# Authenticate with Kaggle (if needed)
# kagglehub.login()

# Download VGGFace2-112x112 dataset (cached after first download!)
print("📥 Downloading VGGFace2-112x112 dataset...")
dataset_path = kagglehub.dataset_download("yakhyokhuja/vggface2-112x112")
print(f"✅ Downloaded to: {dataset_path}")

# Access the data directly - no extraction needed!
DATA_DIR = Path(dataset_path)
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

# Verify structure
if TRAIN_DIR.exists():
    train_ids = len([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
    print(f"\n✅ Dataset ready!")
    print(f"   Train identities: {train_ids}")
    print(f"   Train path: {TRAIN_DIR}")

if TEST_DIR.exists():
    test_ids = len([d for d in TEST_DIR.iterdir() if d.is_dir()])
    print(f"   Test identities: {test_ids}")
    print(f"   Test pat

## 🎉 Summary

You've successfully:

1. ✅ Downloaded VGGFace2-112x112 dataset from Kaggle
2. ✅ Verified dataset structure and image quality
3. ✅ Created train/validation split
4. ✅ Compressed with pigz (parallel gzip)
5. ✅ Prepared files for training

### 📤 Next Steps:

1. **Use in Training** - Copy code from Section 9 to your training notebook
2. **Optional: Upload to Kaggle** - Share your processed dataset version
3. **Start Training** - Images are ready for MobileFaceNet training

### ✨ Benefits of VGGFace2-112x112:

- **Pre-processed**: No face detection or alignment needed
- **Perfect Size**: Already 112x112 for MobileFaceNet
- **Large Scale**: 9,131 identities for robust training
- **Fast downloads**: kagglehub caches the dataset locally
- **Ready to Train**: Start training immediately

### ⏱️ Time Comparison:

| Method | First Download | Preprocessing | Total |
|--------|----------------|---------------|-------|
| Original VGGFace2 | 30-60 min | 2-4 hours | 2.5-5 hours |
| VGGFace2-112x112 | 10-30 min | **None needed** | **10-30 min** |

### 📂 Dataset Path:

The dataset is downloaded and cached by kagglehub. Use this code to access it in any notebook:

```python
import kagglehub
path = kagglehub.dataset_download("yakhyokhuja/vggface2-112x112")
print(f"Dataset path: {path}")
```